In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *

In [0]:
passengers_day1 = [
(101,"Rahul Sharma","Hyderabad","Economy","India"),
(102,"Priya Reddy","Bangalore","Business","India"),
(103,"Amit Kumar","Mumbai","Economy","India"),
(104,"Sneha Patel","Delhi","Premium Economy","India"),
(105,"Farhan Ali","Chennai","Economy","India")
]
columns = [
"passenger_id",
"passenger_name",
"city",
"travel_class",
"country"
]
df_day1 = spark.createDataFrame(passengers_day1,columns)

In [0]:
passengers_day2 = [
(102,"Priya Reddy","Bangalore","First Class","India"),
(104,"Sneha Patel","Hyderabad","Premium Economy","India"),
(106,"Neha Singh","Pune","Economy","India"),
(107,"Arjun Verma","Kochi","Business","India")
]

df_day2 = spark.createDataFrame(passengers_day2,columns)

In [0]:
df_day1.write.format("delta").mode("overwrite").save("/tmp/passengers_delta")

In [0]:
df_day1.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
spark.read.format("delta").load("/tmp/passengers_delta").count()

5

In [0]:
delta_df = spark.read.format("delta") \
    .load("/tmp/passengers_delta")

delta_df.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(spark, "/tmp/passengers_delta")

deltaTable.history().show()

+-------+-------------------+---------------+--------------------+---------+--------------------+----+-----------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|         userId|            userName|operation| operationParameters| job|         notebook|queryHistoryStatementId|           clusterId|readVersion|   isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+---------------+--------------------+---------+--------------------+----+-----------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|      9|2026-06-17 12:19:11|148916151530801|azuser7222_mml.lo...|    WRITE|{mode -> Overwrit...|NULL|{284788352675044}|   86d29c5a-74b9-41c...|0617-115947-l7nmx...|          8|WriteSerializable|        fa

In [0]:
deltaTable.alias("target") \
.merge(
    df_day2.alias("source"),
    "target.passenger_id = source.passenger_id"
).whenMatchedUpdate(set={
    "passenger_name":"source.passenger_name",
    "city":"source.city",
    "travel_class":"source.travel_class",
    "country":"source.country"
}).whenNotMatchedInsert(values={
    "passenger_id":"source.passenger_id",
    "passenger_name":"source.passenger_name",
    "city":"source.city",
    "travel_class":"source.travel_class",
    "country":"source.country"
}) \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.read.format("delta").load("/tmp/passengers_delta").filter(col("passenger_id")==102).show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore| First Class|  India|
+------------+--------------+---------+------------+-------+



In [0]:
spark.read.format("delta").load("/tmp/passengers_delta").filter(col("passenger_id")==106).show()

+------------+--------------+----+------------+-------+
|passenger_id|passenger_name|city|travel_class|country|
+------------+--------------+----+------------+-------+
|         106|    Neha Singh|Pune|     Economy|  India|
+------------+--------------+----+------------+-------+



In [0]:
version0 =spark.read.format("delta").option("versionAsOf",0).load("/tmp/passengers_delta")

version0.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
latest = spark.read.format("delta").load("/tmp/passengers_delta")

latest.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
print("Version 0")
version0.show()

print("Latest Version")
latest.show()

Version 0
+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+

Latest Version
+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         102|

In [0]:
version0.filter(col("passenger_id")==102).show()

latest.filter(col("passenger_id")==102).show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore|    Business|  India|
+------------+--------------+---------+------------+-------+

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore| First Class|  India|
+------------+--------------+---------+------------+-------+



In [0]:
version0.filter(col("passenger_id")==104).show()

latest.filter(col("passenger_id")==104).show()

+------------+--------------+-----+---------------+-------+
|passenger_id|passenger_name| city|   travel_class|country|
+------------+--------------+-----+---------------+-------+
|         104|   Sneha Patel|Delhi|Premium Economy|  India|
+------------+--------------+-----+---------------+-------+

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:

%sql OPTIMIZE '/tmp/passengers_delta'


path,metrics
dbfs:/tmp/passengers_delta,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1781698956743, 1781698957191, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


In [0]:
%sql OPTIMIZE '/tmp/passengers_delta'
ZORDER BY (city)

path,metrics
dbfs:/tmp/passengers_delta,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 2027), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781698943820, 1781698944419, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


In [0]:
deltaTable.delete(col("passenger_id")==105).show()

DataFrame[num_affected_rows: bigint]

In [0]:
%sql
     DESCRIBE HISTORY '/tmp/passengers_delta'


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
11,2026-06-17T12:19:21.000Z,148916151530801,azuser7222_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(284788352675044),87b32af2-2e47-4abf-94ab-0ffb71747624,0617-115947-l7nmxm00-v2n,10,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 8941, p25FileSize -> 2027, numDeletionVectorsRemoved -> 1, minFileSize -> 2027, numAddedFiles -> 1, maxFileSize -> 2027, p75FileSize -> 2027, p50FileSize -> 2027, numAddedBytes -> 2027)",null,Databricks-Runtime/18.2.x-photon-scala2.13
10,2026-06-17T12:19:19.000Z,148916151530801,azuser7222_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(passenger_id#14929L = passenger_id#14954L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(284788352675044),87b32af2-2e47-4abf-94ab-0ffb71747624,0617-115947-l7nmxm00-v2n,9,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 6974, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 2681, materializeSourceTimeMs -> 90, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1070, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1488)",null,Databricks-Runtime/18.2.x-photon-scala2.13
9,2026-06-17T12:19:11.000Z,148916151530801,azuser7222_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(284788352675044),86d29c5a-74b9-41cd-8f54-42bc843bedb7,0617-115947-l7nmxm00-v2n,8,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1967, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1967)",null,Databricks-Runtime/18.2.x-photon-scala2.13
8,2026-06-17T12:19:06.000Z,148916151530801,azuser7222_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(284788352675044),a88cc347-806c-4526-87d3-794d5a431f10,0617-115947-l7nmxm00-v2n,7,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1999, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1967)",null,Databricks-Runtime/18.2.x-photon-scala2.13
7,2026-06-17T12:13:36.000Z,148916151530801,azuser7222_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(passenger_id#13965L = 105)""])",null,List(284788352675044),711b23da-e8cc-48f0-8a12-892bd0670cf4,0617-115947-l7nmxm00-v2n,6,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 519, numDeletionVectorsUpdated -> 0, numDeletedRows -> 0, scanTimeMs -> 516, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/18.2.x-photon-scala2.13
6,2026-06-17T12:13:12.000Z,148916151530801,azuser7222_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(284788352675044),f3af6529-ff24-4a33-b87e-07e6661ea851,0617-115947-l7nmxm00-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2027, p25FileSize -> 1999, numDeletionVectorsRemoved -> 1, minFileSize -> 1999, numA

In [0]:
%sql VACUUM '/tmp/passengers_delta'
RETAIN 168 HOURS


path
dbfs:/tmp/passengers_delta
